<a href="https://colab.research.google.com/github/eli576/USFQ_Python/blob/main/Taller_CC_Deber_05_clean.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install deap
from deap import creator, base, tools

import random
import time


In [ ]:
#initialising for creation of 8-bit chromosome

chrom_length = 8

toolbox = base.Toolbox()
toolbox.register("one_to_eight", random.randint, 0, 7)

#define fitness function strategy - minimizing conflicts
creator.create("FitnessMin", base.Fitness, weights=(-1.0,))

#create individuals of population
creator.create("Individual", list, fitness=creator.FitnessMin)

#use class previously defined to create each chromosome
toolbox.register("individualCreator", tools.initRepeat, creator.Individual, toolbox.one_to_eight, chrom_length)

#create population
toolbox.register("populationCreator", tools.initRepeat, list, toolbox.individualCreator)

In [ ]:
#fitness function

def eight_queen_fitness(individual):

    conflicts = 0
    n = 8

    # Iterate through all possible pairs of queens
    for i in range(n):
        for j in range(i + 1, n):
            row_i = individual[i]
            col_i = i
            row_j = individual[j]
            col_j = j

            # Check for queens in the same row
            if row_i == row_j:
                conflicts += 1
            # Check for queens in the same diagonal
            elif abs(row_i - row_j) == abs(col_i - col_j):
                conflicts += 1
    # Lower conflict count is better - minimisation (higher fitness for minimization by DEAP's convention of -1.0 weight for FitnessMin)
    return (conflicts,)


#define fitness operator
toolbox.register("evaluate", eight_queen_fitness)

In [ ]:
population_sizes = [30, 90, 160]
p_mutations = [0.05, 0.1, 0.3]
p_crossovers = [0.6, 0.75, 0.9]
selection_methods = ['torneo', 'ruleta', 'rango']
crossover_types = ['PMX', 'OX', 'uniforme']
max_generations_list = [200, 600, 1000]

num_replicates = 3  #runs per configuration



#parameter configuration
def configure_operators(selection_method, crossover_type):

    # Selection
    if selection_method == 'torneo':
        toolbox.register("select", tools.selTournament, tournsize=3)
    elif selection_method == 'ruleta':
        toolbox.register("select", tools.selRoulette)
    elif selection_method == 'rango':
        toolbox.register("select", tools.selStochasticUniversalSampling)

    # Crossover
    if crossover_type == 'PMX':
        toolbox.register("mate", tools.cxPartialyMatched)
    elif crossover_type == 'OX':
        toolbox.register("mate", tools.cxOrdered)
    elif crossover_type == 'uniforme':
        toolbox.register("mate", tools.cxUniform, indpb=0.5)

    # Mutation (fixed operator, prob will be passed to GA)
    toolbox.register("mutate", tools.mutShuffleIndexes, indpb=0.2)



# run genetic algorithm for one configuration
def run_ga_8_queens(population_size, p_crossover, p_mutation, max_generations):

    # Create initial population
    population = toolbox.populationCreator(n=population_size)

    # Evaluate initial population
    for ind in population:
        ind.fitness.values = toolbox.evaluate(ind)

    start_time = time.time()

    best_ind = tools.selBest(population, k=1)[0]
    best_conflicts = best_ind.fitness.values[0]
    generation = 0
    converged = False

    for gen in range(1, max_generations + 1):
        generation = gen

        # Selection
        offspring = toolbox.select(population, len(population))
        offspring = list(map(toolbox.clone, offspring))

        # Crossover
        for i in range(0, len(offspring), 2):
            if i + 1 < len(offspring) and random.random() < p_crossover:
                toolbox.mate(offspring[i], offspring[i + 1])
                del offspring[i].fitness.values
                del offspring[i + 1].fitness.values

        # Mutation
        for mutant in offspring:
            if random.random() < p_mutation:
                toolbox.mutate(mutant)
                del mutant.fitness.values

        # Re-evaluate mutated / crossed offspring
        invalid_ind = [ind for ind in offspring if not ind.fitness.valid]
        for ind in invalid_ind:
            ind.fitness.values = toolbox.evaluate(ind)

        population = offspring

        # Track best individual
        current_best = tools.selBest(population, k=1)[0]
        if current_best.fitness.values[0] < best_conflicts:
            best_ind = toolbox.clone(current_best)
            best_conflicts = best_ind.fitness.values[0]

        # Early stopping if we found a solution with 0 conflicts
        if best_conflicts == 0:
            break

    exec_time = time.time() - start_time

    result = {
        "population_size": population_size,
        "p_crossover": p_crossover,
        "p_mutation": p_mutation,
        "max_generations": max_generations,
        "generations_used": generation,
        "best_conflicts": best_conflicts,
        "found_optimal": (best_conflicts == 0),
        "best_individual": best_ind,
        "execution_time_sec": exec_time,
    }
    return result


# Run for all configurations/ full experiment
all_results = []
config_id = 0

for pop_size in population_sizes:
    for p_mut in p_mutations:
        for p_cx in p_crossovers:
            for sel_method in selection_methods:
                for cx_type in crossover_types:
                    for max_gens in max_generations_list:
                        config_id += 1
                        print(f"Config {config_id}")

                        # Configure DEAP operators for this configuration
                        configure_operators(sel_method, cx_type)

                        # Run several replicates for this configuration
                        for rep in range(1, num_replicates + 1):
                            #print(f"  - Replicate {rep}")
                            res = run_ga_8_queens(
                                population_size=pop_size,
                                p_crossover=p_cx,
                                p_mutation=p_mut,
                                max_generations=max_gens
                            )
                            # Add metadata about configuration and replicate
                            res["config_id"] = config_id
                            res["replicate"] = rep
                            res["selection_method"] = sel_method
                            res["crossover_type"] = cx_type

                            all_results.append(res)

# Convert all results to a DataFrame
results_df = pd.DataFrame(all_results)
display(results_df)


Config 1
Config 2
Config 3
Config 4
Config 5
Config 6
Config 7
Config 8
Config 9
Config 10
Config 11
Config 12
Config 13
Config 14
Config 15
Config 16
Config 17
Config 18
Config 19
Config 20
Config 21
Config 22
Config 23
Config 24
Config 25
Config 26
Config 27
Config 28
Config 29
Config 30
Config 31
Config 32
Config 33
Config 34
Config 35
Config 36
Config 37
Config 38
Config 39
Config 40
Config 41
Config 42
Config 43
Config 44
Config 45
Config 46
Config 47
Config 48
Config 49
Config 50
Config 51
Config 52
Config 53
Config 54
Config 55
Config 56
Config 57
Config 58
Config 59
Config 60
Config 61
Config 62
Config 63
Config 64
Config 65
Config 66
Config 67
Config 68
Config 69
Config 70
Config 71
Config 72
Config 73
Config 74
Config 75
Config 76
Config 77
Config 78
Config 79
Config 80
Config 81
Config 82
Config 83
Config 84
Config 85
Config 86
Config 87
Config 88
Config 89
Config 90
Config 91
Config 92
Config 93
Config 94
Config 95
Config 96
Config 97
Config 98
Config 99
Config 100
Config 1

,population_size,p_crossover,p_mutation,max_generations,generations_used,best_conflicts,found_optimal,best_individual,execution_time_sec,config_id,replicate,selection_method,crossover_type
0,30,0.6,0.05,200,200,2.0,False,"[6, 3, 7, 0, 3, 5, 2, 4]",0.143626,1,1,torneo,PMX
1,30,0.6,0.05,200,9,0.0,True,"[3, 1, 7, 5, 0, 2, 4, 6]",0.005479,1,2,torneo,PMX
2,30,0.6,0.05,200,10,0.0,True,"[2, 5, 7, 1, 3, 0, 6, 4]",0.005956,1,3,torneo,PMX
3,30,0.6,0.05,600,600,1.0,False,"[5, 2, 5, 7, 0, 3, 6, 4]",0.583828,2,1,torneo,PMX
4,30,0.6,0.05,600,66,0.0,True,"[2, 5, 1, 6, 4, 0, 7, 3]",0.038700,2,2,torneo,PMX
...,...,...,...,...,...,...,...,...,...,...,...,...,...
2182,160,0.9,0.30,600,600,2.0,False,"[3, 6, 1, 5, 0, 6, 4, 2]",6.690264,728,2,rango,uniforme
2183,160,0.9,0.30,600,600,1.0,False,"[6, 2, 2, 5, 7, 4, 1, 3]",5.582489,728,3,rango,uniforme
2184,160,0.9,0.30,1000,1000,1.0,False,"[2, 6, 1, 3, 0, 4, 7, 5]",10.398267,729,1,rango,uniforme
2185,160,0.9,0.30,1000,1000,2.0,False,"[2, 4, 6, 3, 0, 2, 1, 5]",10.331903,729,2,rango,uniforme


Mounted at /content/drive


OSError: Cannot save file into a non-existent directory: '/content/drive/MyDrive/GA_Results'

In [5]:

import pandas as pd
from google.colab import drive

drive.mount('/content/drive')


!cp "/content/drive/MyDrive/FilesColab/ga_results_8queens.csv" /content/
data = open('ga_results_8queens.csv')



results_df = pd.read_csv(data)

print(results_df.shape)
results_df.head()



Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
(2187, 13)


,population_size,p_crossover,p_mutation,max_generations,generations_used,best_conflicts,found_optimal,best_individual,execution_time_sec,config_id,replicate,selection_method,crossover_type
0,30,0.6,0.05,200,200,2.0,False,"[6, 3, 7, 0, 3, 5, 2, 4]",0.143626,1,1,torneo,PMX
1,30,0.6,0.05,200,9,0.0,True,"[3, 1, 7, 5, 0, 2, 4, 6]",0.005479,1,2,torneo,PMX
2,30,0.6,0.05,200,10,0.0,True,"[2, 5, 7, 1, 3, 0, 6, 4]",0.005956,1,3,torneo,PMX
3,30,0.6,0.05,600,600,1.0,False,"[5, 2, 5, 7, 0, 3, 6, 4]",0.583828,2,1,torneo,PMX
4,30,0.6,0.05,600,66,0.0,True,"[2, 5, 1, 6, 4, 0, 7, 3]",0.038700,2,2,torneo,PMX


In [17]:
import pandas as pd

# 1) configuration summary
summary = results_df.groupby("config_id").agg(
    population_size=("population_size", "first"),
    p_crossover=("p_crossover", "first"),
    p_mutation=("p_mutation", "first"),
    selection_method=("selection_method", "first"),
    crossover_type=("crossover_type", "first"),
    max_generations=("max_generations", "first"),

    # performance metrics
    success_rate=("found_optimal", "mean"), # fraction of runs that found a solution
    avg_best_conflicts=("best_conflicts", "mean"),
    avg_time=("execution_time_sec", "mean"),
    avg_generations=("generations_used", "mean"),

    # variability (for stability)
    std_time=("execution_time_sec", "std"),
).reset_index()

summary.fillna(0, inplace=True)



# 2) stability: A configuration is stable if its runtimes are similar across runs.
stable_mask = summary["std_time"] < (summary["avg_time"] * 0.30)   # time variation < 30%

stable_configs   = summary[stable_mask]
unstable_configs = summary[~stable_mask]



# 3) Top 10 best configurations that are stable
#    Ranking criteria:
#      1) higher success_rate
#      2) lower avg_best_conflicts
#      3) lower avg_time
best_stable_top10 = stable_configs.sort_values(
    by=["success_rate", "avg_best_conflicts", "avg_time"],
    ascending=[False, True, True]
).head(10)

print("\n1) TOP 10 BEST (STABLE) CONFIGURATIONS")
display(best_stable_top10)


# 4) Unstable configurations filtered out from top performers

top10_best_overall = summary.sort_values(
    by=["success_rate", "avg_best_conflicts", "avg_time"],
    ascending=[False, True, True]
).head(10)

# unstable among those top 10 best
unstable_filtered_out = top10_best_overall[
    top10_best_overall["config_id"].isin(unstable_configs["config_id"])
]

print("\n2a) UNSTABLE CONFIGURATIONS FILTERED OUT FROM TOP 10 BEST")
display(unstable_filtered_out)

# Visualise runtimes for unstable configurations
unstable_ids = unstable_filtered_out["config_id"].unique()

unstable_runtime_details = results_df[
    results_df["config_id"].isin(unstable_ids)
].sort_values(
    by=["config_id", "replicate"]
)

unstable_runtime_details = unstable_runtime_details[
    ["config_id", "replicate", "execution_time_sec"]
]

print("\n2b) RUNTIMES OF EACH REPLICATION FOR THESE UNSTABLE CONFIGURATIONS")
display(unstable_runtime_details)


# 5) Top 10 worst configurations (regardless of stability)
#    Ranking criteria (inverted):
#      1) lower success_rate
#      2) higher avg_best_conflicts
#      3) higher avg_time
worst_top10 = summary.sort_values(
    by=["success_rate", "avg_best_conflicts", "avg_time"],
    ascending=[True, False, False]
).head(10)

print("\n3) TOP 10 WORST CONFIGURATIONS")
display(worst_top10)



1) TOP 10 BEST (STABLE) CONFIGURATIONS


,config_id,population_size,p_crossover,p_mutation,selection_method,crossover_type,max_generations,success_rate,avg_best_conflicts,avg_time,avg_generations,std_time
648,649,160,0.60,0.30,torneo,PMX,200,1.0,0.0,0.012764,3.666667,0.002202
299,300,90,0.90,0.05,torneo,PMX,1000,1.0,0.0,0.014664,7.333333,0.004052
378,379,90,0.90,0.10,torneo,PMX,200,1.0,0.0,0.014747,7.333333,0.000949
676,677,160,0.75,0.30,torneo,PMX,600,1.0,0.0,0.015951,3.666667,0.002867
406,407,90,0.60,0.30,torneo,PMX,600,1.0,0.0,0.017216,9.333333,0.002242
540,541,160,0.90,0.05,torneo,PMX,200,1.0,0.0,0.019235,5.333333,0.002188
677,678,160,0.75,0.30,torneo,PMX,1000,1.0,0.0,0.019237,5.333333,0.002575
432,433,90,0.75,0.30,torneo,PMX,200,1.0,0.0,0.020430,10.000000,0.000767
272,273,90,0.75,0.05,torneo,PMX,1000,1.0,0.0,0.021839,11.333333,0.006424
623,624,160,0.90,0.10,torneo,PMX,1000,1.0,0.0,0.021955,6.000000,0.004156



2a) UNSTABLE CONFIGURATIONS FILTERED OUT FROM TOP 10 BEST


,config_id,population_size,p_crossover,p_mutation,selection_method,crossover_type,max_generations,success_rate,avg_best_conflicts,avg_time,avg_generations,std_time
216,217,30,0.90,0.30,torneo,PMX,200,1.0,0.0,0.011407,16.333333,0.006182
56,57,30,0.90,0.05,torneo,PMX,1000,1.0,0.0,0.011738,17.666667,0.008026
433,434,90,0.75,0.30,torneo,PMX,600,1.0,0.0,0.013042,6.666667,0.004083
352,353,90,0.75,0.10,torneo,PMX,600,1.0,0.0,0.014810,7.666667,0.008274
270,271,90,0.75,0.05,torneo,PMX,200,1.0,0.0,0.014985,7.333333,0.008697
298,299,90,0.90,0.05,torneo,PMX,600,1.0,0.0,0.015406,7.666667,0.011180
541,542,160,0.90,0.05,torneo,PMX,600,1.0,0.0,0.015512,4.333333,0.007420



2b) RUNTIMES OF EACH REPLICATION FOR THESE UNSTABLE CONFIGURATIONS


,config_id,replicate,execution_time_sec
168,57,1,0.005147
169,57,2,0.009392
170,57,3,0.020676
648,217,1,0.010844
649,217,2,0.005527
650,217,3,0.017851
810,271,1,0.024502
811,271,2,0.013004
812,271,3,0.007449
894,299,1,0.006052



3) TOP 10 WORST CONFIGURATIONS


,config_id,population_size,p_crossover,p_mutation,selection_method,crossover_type,max_generations,success_rate,avg_best_conflicts,avg_time,avg_generations,std_time
203,204,30,0.75,0.30,ruleta,OX,1000,0.0,4.000000,0.855942,1000.0,0.008667
188,189,30,0.60,0.30,rango,uniforme,1000,0.0,4.000000,0.746396,1000.0,0.013197
211,212,30,0.75,0.30,rango,OX,600,0.0,4.000000,0.510303,600.0,0.005838
175,176,30,0.60,0.30,ruleta,OX,600,0.0,4.000000,0.486453,600.0,0.009446
94,95,30,0.60,0.10,ruleta,OX,600,0.0,4.000000,0.477992,600.0,0.011178
156,157,30,0.90,0.10,rango,OX,200,0.0,4.000000,0.173526,200.0,0.004503
39,40,30,0.75,0.05,ruleta,OX,200,0.0,4.000000,0.162373,200.0,0.003624
183,184,30,0.60,0.30,rango,OX,200,0.0,4.000000,0.160581,200.0,0.001949
311,312,90,0.90,0.05,ruleta,OX,1000,0.0,3.666667,4.211627,1000.0,0.681321
77,78,30,0.90,0.05,rango,OX,1000,0.0,3.666667,1.246256,1000.0,0.360010
